# QTrack 2026 — Level 0: Quantum circuits & measurement

**Tri-Co Hackathon · Bi-Co Quantum Computing Club**

**Mission:** Prepare simple quantum states, apply gates, predict measurement outcomes, and understand shot noise.

**Prerequisites:** Basic Python (functions and dictionaries). No quantum mechanics needed.  
**Deliverable:** A completed `.ipynb` notebook with your code, results, and explanations.

Follow the numbered tasks, implement the `TODO` functions, and use the self-checks to test your work.


## Getting started

Run the next cell. In Google Colab, upload this notebook and choose **Runtime → Run all** after completing your tasks. The first run may install Python packages using the internet. After installation, all circuit simulations run locally on the CPU.

**Bit order convention:** Qiskit writes the most significant bit on the **left** of a bitstring. For example `"10"` means qubit 1 is `1` and qubit 0 is `0`. Our checks follow this convention.

**If you get `ModuleNotFoundError`:** rerun the installation cell or restart the notebook kernel and rerun.

In [125]:
# First run: installs dependencies only if missing (requires internet).
import sys, subprocess, importlib.util
if importlib.util.find_spec("qiskit") is None or importlib.util.find_spec("qiskit_aer") is None or importlib.util.find_spec("qiskit_ibm_runtime") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "qiskit>=2.5,<3", "qiskit-aer>=0.17,<0.19", "qiskit-ibm-runtime", "numpy>=1.26,<3", "matplotlib>=3.8"])
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService
print("Qiskit ready. Free local simulation only; no IBM account or paid hardware required.")

Qiskit ready. Free local simulation only; no IBM account or paid hardware required.


### How to work
1. Read the task and its expected result.
2. Replace `return None` with your implementation.
3. Run the **self-check** underneath. A passing check verifies an observable property but is not a complete grading decision.
4. Answer the **Explain** questions in Markdown, using your own words.
5. Run the notebook top to bottom before exporting. Do not delete your code, plots or outputs.

## Worked example — the |+⟩ state

The Hadamard gate creates a superposition from |0⟩. Before measurement, the ideal probability of 0 and 1 is 50% each. Here we use an exact statevector simulation.

In [126]:
demo = QuantumCircuit(1)
demo.h(0)
print(demo.draw(output="text"))
print(Statevector.from_instruction(demo).probabilities_dict())

   ┌───┐
q: ┤ H ├
   └───┘
{np.str_('0'): np.float64(0.4999999999999999), np.str_('1'): np.float64(0.4999999999999999)}


### Task 0A — Basis states (20 min)

Make a one- or two-qubit basis state. **Expected:** preparing `10` produces probability 1 for `10` and zero elsewhere.

In [127]:
def prepare_basis_state(bits: str) -> QuantumCircuit:
    """Create |bits⟩ using X gates. Input: '0','1','00','01','10','11'. Use Qiskit bit order; raise ValueError for any other input."""
    basis = QuantumCircuit(len(bits))
    for i, bit in enumerate(bits):
        if bit == '1':
            basis.x(len(bits) - i - 1)
    return basis

**Self-check:** run after implementing your function.

In [128]:
# Visible self-check (not a hidden grader)
if prepare_basis_state("10") is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    probs = Statevector.from_instruction(prepare_basis_state("10")).probabilities_dict()
    assert abs(probs.get("10", 0) - 1) < 1e-10
    print("Self-check passed.")

Self-check passed.


### Task 0B — Superposition (10 min)

Prepare |+⟩. **Expected:** probabilities of 0 and 1 are each 0.5.

In [129]:
def make_plus() -> QuantumCircuit:
    """Return a 1-qubit circuit preparing |+⟩ from |0⟩."""
    circuit = QuantumCircuit(1)
    circuit.h(0)
    return circuit

**Self-check:** run after implementing your function.

In [130]:
# Visible self-check (not a hidden grader)
if make_plus() is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    p = Statevector.from_instruction(make_plus()).probabilities_dict()
    assert abs(p.get("0", 0)-.5) < 1e-10 and abs(p.get("1", 0)-.5) < 1e-10
    print("Self-check passed.")

Self-check passed.


### Task 0C — Detecting phase (15 min)

Apply H, Z, H in that order. **Expected:** the final measurement gives `1` with certainty.

In [131]:
def phase_flip_detector() -> QuantumCircuit:
    """Create a circuit applying H, then Z, then H to |0⟩ (no measurements)."""
    flip = QuantumCircuit(1)
    flip.h(0)
    flip.z(0)
    flip.h(0)
    return flip

**Self-check:** run after implementing your function.

In [132]:
# Visible self-check (not a hidden grader)
if phase_flip_detector() is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    p = Statevector.from_instruction(phase_flip_detector()).probabilities_dict()
    assert p.get("1",0) > .999999
    print("Self-check passed.")

Self-check passed.


### Task 0D — Sampling versus exact probabilities (20 min)

Measure |+⟩ repeatedly using `AerSimulator`. **Expected:** the counts sum to `shots` and approach a 50/50 split as shots increase.

In [133]:
def sample_plus(shots: int = 1024, seed: int = 7) -> dict:
    """Measure |+⟩ with the given number of shots and random seed; return counts dict."""
    plus = QuantumCircuit(1)
    plus.h(0)
    plus.measure_all()

    backend = AerSimulator(seed_simulator=seed)

    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    isa_circuit = pm.run(plus)

    sampler = Sampler(mode=backend)

    job = sampler.run([isa_circuit], shots=shots)
    result = job.result()

    return result[0].data.meas.get_counts()


**Self-check:** run after implementing your function.

In [134]:
# Visible self-check (not a hidden grader)
if sample_plus() is None:
    print("Not completed yet. Replace the TODO and rerun.")
else:
    counts = sample_plus(shots=1024, seed=7)
    assert sum(counts.values()) == 1024
    assert abs(counts.get("0",0)/1024-.5) < .12
    print("Self-check passed.")

Self-check passed.


## Results and explanation — required

**0E (written response, approx. 100–180 words total)**

1. What changes when you increase the number of shots from 128 to 4096? What does not change?
2. Why does Z not visibly change the probabilities of |+⟩ immediately, while H–Z–H produces a deterministic result?
3. Show your circuit diagram for Task 0C and report the simulated probabilities and at least two measured distributions.

1) Increasing the number of shots from 128 to 4096 increases the amount of measurements taken and consequentially causes the average of the measurements to be more likely to be closer to the theoretical probabilities, essentially increasing the total accuracy of the net measurement. What does not change is the theoretical probabilities.

**2. Why Z leaves the probabilities unchanged, but H–Z–H gives a deterministic result**

In the computational basis, measurement probabilities are the squared magnitudes of the amplitudes. Z changes the relative phase of $|+\rangle$ by flipping the sign of its $|1\rangle$ amplitude:

$$
H=\frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&-1\end{pmatrix},\qquad
Z=\begin{pmatrix}1&0\\0&-1\end{pmatrix},\qquad
|0\rangle=\begin{pmatrix}1\\0\end{pmatrix}.
$$

$$
H|0\rangle=\frac{1}{\sqrt{2}}\begin{pmatrix}1\\1\end{pmatrix}=|+\rangle,
\qquad
Z|+\rangle=\frac{1}{\sqrt{2}}\begin{pmatrix}1&0\\0&-1\end{pmatrix}\begin{pmatrix}1\\1\end{pmatrix}
=\frac{1}{\sqrt{2}}\begin{pmatrix}1\\-1\end{pmatrix}=|-\rangle.
$$

Both states give $P(0)=P(1)=1/2$, since $|1/\sqrt{2}|^2=|-1/\sqrt{2}|^2=1/2$. The final H makes the amplitudes interfere:

$$
HZH|0\rangle=H|-\rangle
=\frac{1}{2}\begin{pmatrix}1&1\\1&-1\end{pmatrix}\begin{pmatrix}1\\-1\end{pmatrix}
=\frac{1}{2}\begin{pmatrix}1-1\\1+1\end{pmatrix}
=\begin{pmatrix}0\\1\end{pmatrix}=|1\rangle.
$$

The contributions to $|0\rangle$ cancel, while those to $|1\rangle$ add. Thus, starting from $|0\rangle$, H–Z–H gives $P(1)=1$.

3) H-Z-H circuit:
```text
   ┌───┐┌───┐┌───┐
q: ┤ H ├┤ Z ├┤ H ├
   └───┘└───┘└───┘
```

Simulated probabilities:
```text
Seed 7: {'1': 1024}
Seed 8: {'1': 1024}
```

In [143]:
# Evidence: run after finishing tasks
if make_plus() is not None and phase_flip_detector() is not None and sample_plus() is not None:
    print("H-Z-H circuit:\n", phase_flip_detector().draw(output="text"))
    print("Exact H-Z-H probabilities:", Statevector.from_instruction(phase_flip_detector()).probabilities_dict())
    print("128 shots:", sample_plus(128, seed=7))
    print("4096 shots:", sample_plus(4096, seed=7))
else:
    print("Finish Tasks 0B, 0C, and 0D for the evidence output.")

plus = phase_flip_detector()
plus.measure_all()

simulator = AerSimulator(seed_simulator=7)

pm = generate_preset_pass_manager(backend=simulator, optimization_level=1)
isa_circuit = pm.run(plus)

sampler = Sampler(mode=simulator)

job = sampler.run([isa_circuit], shots=1024)
result = job.result()

print(result[0].data.meas.get_counts())

simulator = AerSimulator(seed_simulator=8)

pm = generate_preset_pass_manager(backend=simulator, optimization_level=1)
isa_circuit = pm.run(plus)

sampler = Sampler(mode=simulator)

job = sampler.run([isa_circuit], shots=1024)
result = job.result()

print(result[0].data.meas.get_counts())

H-Z-H circuit:
    ┌───┐┌───┐┌───┐
q: ┤ H ├┤ Z ├┤ H ├
   └───┘└───┘└───┘
Exact H-Z-H probabilities: {np.str_('1'): np.float64(0.9999999999999996)}
128 shots: {'0': 58, '1': 70}
4096 shots: {'0': 1968, '1': 2128}
{'1': 1024}
{'1': 1024}


## Level 0 completion checklist
- [X] Tasks 0A–0D implemented (no placeholder returns)
- [X] Self-checks pass
- [X] Two count distributions reported
- [X] Task 0E explanation written
- [X] Notebook runs in order without errors

**Optional stretch:** Which gates produce the same final measurement probabilities but different phases? Provide an example and explain.

**Answer:** Diagonal phase gates such as Z, S, and T preserve computational-basis measurement probabilities while changing relative phases. For example, applying I or Z to $|+\rangle$ produces

$$
I|+\rangle=\frac{|0\rangle+|1\rangle}{\sqrt{2}},\qquad
Z|+\rangle=\frac{|0\rangle-|1\rangle}{\sqrt{2}}.
$$

Both give 0 and 1 with probability $1/2$, but their relative phases are 0 and $\pi$. This is not just a global phase: applying H afterward gives $|0\rangle$ and $|1\rangle$, respectively, making the difference measurable.

## AI Use Disclosure
Task 0D:
- learning how to perform random sampling with the imports provided, as the resources in concepts_and_links asked for other imports
Task 0E:
- fixing circuit diagram markdown formatting issues
- use ChatGPT to write LaTex code for matrix multiplication